# Modeling Total International Arrivals

This notebook compares Holt-Winters, SARIMA, and SARIMA-GARCH for the primary target: total international arrivals.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.config import CLEAN_SEGMENTS, FIGURES, TABLES, TARGET_COLUMNS, SEGMENT_COLUMNS, SERIES_COLORS, SERIES_LABELS
from src.plotting import annotate_events, save_figure, set_academic_style

set_academic_style()
df = pd.read_csv(CLEAN_SEGMENTS, parse_dates=["date"]).set_index("date").sort_index()
df.index.freq = "MS"

from statsmodels.graphics.tsaplots import plot_acf
from scipy import stats
from src.models.evaluation import evaluate_target, split_series

## Train/Test Split

Observation: the test period covers the recovery phase from 2023 to 2025. Statistical implication: out-of-sample evaluation is intentionally difficult because the structure differs from the pre-2023 training sample. Tourism implication: forecast errors measure recovery uncertainty, not only model weakness.

In [ ]:
target = "international_arrivals"
train, test = split_series(df[target])
train.index.min(), train.index.max(), test.index.min(), test.index.max(), len(train), len(test)

## Model Estimation and Forecasts

Observation: each method extrapolates seasonality differently. Statistical implication: accuracy comparisons reveal whether smoothing, SARIMA dependence, or residual volatility modeling is more useful. Tourism implication: the preferred model should support planning under post-COVID uncertainty.

In [ ]:
rows, results, forecasts = evaluate_target(df[target], target)
metrics = pd.DataFrame(rows)
metrics.to_csv(TABLES / "model_metrics_total.csv", index=False)
metrics

In [ ]:
fig, ax = plt.subplots()
ax.plot(df.loc["2018":].index, df.loc["2018":, target], color=SERIES_COLORS[target], label="Observed")
for model in forecasts["model"].unique():
    sub = forecasts[forecasts["model"] == model].copy()
    sub["date"] = pd.to_datetime(sub["date"])
    ax.plot(sub["date"], sub["forecast"], label=model, alpha=0.85)
ax.set_title("Out-of-Sample Forecasts for Total Arrivals")
ax.set_ylabel("Monthly arrivals")
ax.legend(fontsize=8)
save_figure(fig, FIGURES / "12_total_model_comparison.png")
plt.show()

## Confidence Intervals and Residual Diagnostics

Observation: forecast intervals widen under residual uncertainty. Statistical implication: residual autocorrelation and volatility clustering indicate remaining structure. Tourism implication: decision makers should use ranges for staffing and capacity decisions.

In [ ]:
best_model = metrics[metrics["status"].eq("ok")].sort_values("sMAPE").iloc[0]["model"]
best = results[best_model]
best_fc = forecasts[forecasts["model"].eq(best_model)].copy()
best_fc["date"] = pd.to_datetime(best_fc["date"])
fig, ax = plt.subplots()
ax.plot(df.loc["2018":].index, df.loc["2018":, target], color=SERIES_COLORS[target], label="Observed")
ax.plot(best_fc["date"], best_fc["forecast"], color="#8f6aa8", label=f"{best_model} forecast")
ax.fill_between(best_fc["date"], best_fc["lower"], best_fc["upper"], color="#8f6aa8", alpha=0.18)
ax.set_title("Best-Model Forecast Interval for Total Arrivals")
ax.set_ylabel("Monthly arrivals")
ax.legend()
save_figure(fig, FIGURES / "10_total_forecast.png")
plt.show()

In [ ]:
resid = best["residuals"].dropna()
fig, axes = plt.subplots(1, 3, figsize=(12, 3.4))
axes[0].plot(resid.index, resid, color=SERIES_COLORS[target])
axes[0].axhline(0, color="#777777", linewidth=0.8)
axes[0].set_title("Residuals")
plot_acf(resid, lags=30, ax=axes[1], zero=False, color=SERIES_COLORS[target])
axes[1].set_title("Residual ACF")
stats.probplot(resid, dist="norm", plot=axes[2])
axes[2].set_title("Normal Q-Q")
for ax in axes:
    ax.grid(False)
save_figure(fig, FIGURES / "13_total_residual_diagnostics.png")
plt.show()

## Interpretation

Observation: the best model minimizes test-period forecast error under recovery volatility. Statistical implication: seasonality is forecastable, but residual shocks remain material. Tourism implication: forecasts should be updated frequently as new VNAT observations arrive.